In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class reservoir_dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        if not root_dir.endswith('/'):
            root_dir += '/'
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len([f for f in os.listdir(self.root_dir) if f.startswith("data_") and f.endswith(".npz")])

    def __getitem__(self, index):
        input_list = []
        output_list = []

        if isinstance(index, int):
            index = [index]

        for idx in index:
            idx = str(idx).rjust(4, "0")
            file_path = self.root_dir + "data_" + idx + ".npz"
            
            with np.load(file_path) as data:
                porosity = data['porosity']
                perm_r = data['perm_r']
                perm_z = data['perm_z']
                pressure_buildup = data['pressure_buildup']
                gas_saturation = data['gas_saturation']
                
                inj_rate = data['inj_rate']
                temperature = data['temperature']
                depth = data['depth']
                Swi = data['Swi']
                lam = data['lam']
                perf_interval = data['perf_interval']

            #  QC & DATA TRACKING 
            # Extract the actual values that violate the [0.0, 1.0] physical bounds
            clipped_gas_vals = gas_saturation[(gas_saturation < 0.0) | (gas_saturation > 1.0)]
            clipped_por_vals = porosity[(porosity < 0.0) | (porosity > 1.0)]

            # Core script checks (NaNs / Infs / Permeability)
            nan_count = sum(np.isnan(arr).sum() for arr in [porosity, perm_r, perm_z, pressure_buildup, gas_saturation])
            inf_count = sum(np.isinf(arr).sum() for arr in [porosity, perm_r, perm_z, pressure_buildup, gas_saturation])

            if nan_count > 0 or inf_count > 0:
                print(f"[WARNING] data_{idx}.npz contains NaNs ({nan_count}) or Infs ({inf_count})!")
            if np.any(perm_r <= 0.0) or np.any(perm_z <= 0.0):
                print(f"[CRITICAL WARNING] data_{idx}.npz has negative or zero permeability!")

            # Execution of clipping
            gas_saturation = np.clip(gas_saturation, 0.0, 1.0)
            porosity = np.clip(porosity, 0.0, 1.0)

            #  DETAILED CLIP LOGGING 
            if len(clipped_gas_vals) > 0 or len(clipped_por_vals) > 0:
                log_msg = f"[QC Log] data_{idx}.npz items clipped:\n"
                if len(clipped_gas_vals) > 0:
                    log_msg += f"  -> Gas Saturation values: {clipped_gas_vals.tolist()}\n"
                if len(clipped_por_vals) > 0:
                    log_msg += f"  -> Porosity values:       {clipped_por_vals.tolist()}\n"
                print(log_msg.strip())
            else:
                print(f"[QC Log] data_{idx}.npz: Clear (0 clipped values).")

            #  Data Tensor Formatting 
            log_perm_r = np.log10(perm_r)
            log_perm_z = np.log10(perm_z)

            input_array = np.zeros((10, 96, 200))
            nz = porosity.shape[0]
            
            input_array[0, :nz, :] = porosity
            input_array[1, :nz, :] = log_perm_r  
            input_array[2, :nz, :] = log_perm_z  
            input_array[3, :, :] = np.ones((96, 200)) * inj_rate
            input_array[4, :, :] = np.ones((96, 200)) * temperature
            input_array[5, :, :] = np.ones((96, 200)) * depth
            input_array[6, :, :] = np.ones((96, 200)) * Swi
            input_array[7, :, :] = np.ones((96, 200)) * lam
            input_array[8, :, :] = np.ones((96, 200)) * perf_interval[0]
            input_array[9, :, :] = np.ones((96, 200)) * perf_interval[1]

            input_tensor = torch.from_numpy(input_array).float()

            output_array = np.zeros((2, 96, 200, 24))
            output_array[0, :nz, :, :] = pressure_buildup
            output_array[1, :nz, :, :] = gas_saturation

            output_tensor = torch.from_numpy(output_array).float()

            input_list.append(input_tensor)
            output_list.append(output_tensor)
            
        sample = torch.stack(input_list, dim=0), torch.stack(output_list, dim=0)

        if len(index) == 1:
            sample = (sample[0].squeeze(0), sample[1].squeeze(0))

        if self.transform:
            sample = self.transform(sample)

        return sample
